[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giovanisaavedra/FIAP/blob/main/ANO%202/FASE%2002/CAP01%20-%20DESAFIO%20INTEGRADOR/classificador_triagem.ipynb)

# 🫀 CardioIA — Fase 2: Diagnóstico Automatizado
## Parte 2: Classificador Básico de Texto para Triagem Clínica (TF-IDF + Scikit-Learn)

**Integrantes:**
- Giovani Saavedra (RM566797)
- Marcio Elifas (RM567871)

**Turma:** 2TIAOR — FIAP ON  
**Disciplina:** IA no Estetoscópio Digital  

### 1. Contexto e Objetivos
Nesta etapa, desenvolvemos um classificador inteligente de texto em linguagem natural para automatizar a triagem clínica de queixas cardiológicas.
O modelo analisa o relato do paciente e prediz o nível de risco:
- **Alto Risco:** Síndromes coronarianas agudas, arritmias graves ou insuficiência cardíaca descompensada.
- **Baixo Risco:** Sintomas musculoesqueléticos, fadiga tensional ou rotina ambulatorial.

In [1]:
# ==============================================================================
# 0. CONFIGURAÇÃO DE AMBIENTE: GOOGLE COLAB vs EXECUÇÃO LOCAL
# ==============================================================================
# Se você estiver executando no Google Colab, a instrução abaixo solicitará
# o upload do arquivo "dataset_triagem.csv" (disponível na pasta dados/ do repositório).
import os
import sys

caminho_csv = os.path.join("dados", "dataset_triagem.csv")

# Verifica se está no Google Colab ou se o arquivo ainda não existe no ambiente
if not os.path.exists(caminho_csv) and not os.path.exists("dataset_triagem.csv"):
    try:
        from google.colab import files
        print("[*] Ambiente Google Colab detectado!")
        print("[!] Por favor, clique no botão abaixo e faça o upload do arquivo dataset_triagem.csv:")
        uploaded = files.upload()
        os.makedirs("dados", exist_ok=True)
        for nome_arq in uploaded.keys():
            destino = os.path.join("dados", nome_arq)
            if not os.path.exists(destino):
                os.rename(nome_arq, destino)
            caminho_csv = destino
            print(f"[+] Arquivo configurado com sucesso em: {caminho_csv}")
    except ImportError:
        # Execução local: procura o dataset no diretório canônico da máquina
        candidatos_locais = [
            "dados/dataset_triagem.csv",
            "/Users/giovanisaavedra/FIAP/FIAP/ANO 2/FASE 02/CAP01 - DESAFIO INTEGRADOR/dados/dataset_triagem.csv",
            os.path.expanduser("~/FIAP/FIAP/ANO 2/FASE 02/CAP01 - DESAFIO INTEGRADOR/dados/dataset_triagem.csv")
        ]
        caminho_csv = next((c for c in candidatos_locais if os.path.exists(c)), caminho_csv)


In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### 2. Carregamento e Exploração da Base de Dados
Carregamos o dataset simulado `dados/dataset_triagem.csv` contendo 50 relatos clínicos anotados e balanceados.

In [2]:
# 1. Carregamento e Exploração da Base de Dados
if not os.path.exists(caminho_csv):
    if os.path.exists("dataset_triagem.csv"):
        caminho_csv = "dataset_triagem.csv"
    else:
        candidatos = [
            "dados/dataset_triagem.csv",
            "/Users/giovanisaavedra/FIAP/FIAP/ANO 2/FASE 02/CAP01 - DESAFIO INTEGRADOR/dados/dataset_triagem.csv",
            os.path.expanduser("~/FIAP/FIAP/ANO 2/FASE 02/CAP01 - DESAFIO INTEGRADOR/dados/dataset_triagem.csv")
        ]
        caminho_csv = next((c for c in candidatos if os.path.exists(c)), "dados/dataset_triagem.csv")

df = pd.read_csv(caminho_csv)
print(f"Dataset carregado de: {caminho_csv}")
print(f"Total de relatos no dataset: {len(df)}")
print("\nDistribuição das classes de risco:")
print(df["risco"].value_counts())
df.head(6)

Dataset carregado de: dados/dataset_triagem.csv
Total de relatos no dataset: 50

Distribuição das classes de risco:
risco
alto risco     25
baixo risco    25
Name: count, dtype: int64


### 3. Divisão Estratificada entre Treino e Teste
Separamos 75% dos dados para treinamento e 25% para teste, garantindo proporções idênticas com `stratify=y`.

In [3]:
X = df["relato"]
y = df["risco"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Amostras de treino: {len(X_treino)}")
print(f"Amostras de teste:  {len(X_teste)}")

Amostras de treino: 37
Amostras de teste:  13


### 4. Vetorização Textual com TF-IDF
Transformamos as frases em matrizes esparsas numéricas utilizando TF-IDF com unigramas e bigramas (`ngram_range=(1, 2)`).

In [4]:
vetorizador = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X_treino_vec = vetorizador.fit_transform(X_treino)
X_teste_vec = vetorizador.transform(X_teste)

print(f"Dimensão da matriz de treino TF-IDF: {X_treino_vec.shape}")
print(f"Total de termos no vocabulário: {len(vetorizador.get_feature_names_out())}")
print(f"Exemplos de termos extraídos: {list(vetorizador.get_feature_names_out())[:12]}")

Dimensão da matriz de treino TF-IDF: (37, 391)
Total de termos no vocabulário: 391
Exemplos de termos extraídos: ['acelerado', 'acelerado batimento', 'acentuado', 'acentuado nos', 'acompanhada', 'acompanhada de', 'aguda', 'aguda no', 'alongamento', 'ao', 'ao apertar', 'ao caminhar']


### 5. Treinamento e Avaliação do Modelo 1: Regressão Logística

In [5]:
modelo_lr = LogisticRegression(random_state=42)
modelo_lr.fit(X_treino_vec, y_treino)
pred_lr = modelo_lr.predict(X_teste_vec)
acc_lr = accuracy_score(y_teste, pred_lr)

print(f"Acurácia Global no Teste: {acc_lr * 100:.2f}%\n")
print("Relatório de Classificação:")
print(classification_report(y_teste, pred_lr))
print("Matriz de Confusão:")
print(confusion_matrix(y_teste, pred_lr))

Acurácia Global no Teste: 92.31%

Relatório de Classificação:
              precision    recall  f1-score   support

  alto risco       1.00      0.86      0.92         7
 baixo risco       0.86      1.00      0.92         6

    accuracy                           0.92        13
   macro avg       0.93      0.93      0.92        13
weighted avg       0.93      0.92      0.92        13

Matriz de Confusão:
[[6 1]
 [0 6]]


> **Análise Específica do Modelo 1 (Regressão Logística):**
> O modelo alcançou **92,31% de acurácia global (12 acertos em 13)**. O seu grande destaque foi a **Precisão de 100% para Alto Risco**, garantindo que todo alerta de gravidade emitido corresponde a um caso real de emergência (zero falsos positivos). O seu único erro foi 1 Falso Negativo (um relato grave classificado como leve, resultando em recall de 86%). A grande vantagem arquitetural da Regressão Logística para sistemas clínicos é a geração de **probabilidades calibradas contínuas (`predict_proba`)** via função sigmoide, permitindo que a equipe médica ajuste dinamicamente o ponto de corte (*threshold*) — por exemplo, alertando para qualquer risco superior a 30% —, eliminando falsos negativos em pronto-socorro.

### 6. Treinamento e Avaliação do Modelo 2: Árvore de Decisão

In [6]:
modelo_dt = DecisionTreeClassifier(random_state=42, max_depth=5)
modelo_dt.fit(X_treino_vec, y_treino)
pred_dt = modelo_dt.predict(X_teste_vec)
acc_dt = accuracy_score(y_teste, pred_dt)

print(f"Acurácia Global no Teste: {acc_dt * 100:.2f}%\n")
print("Relatório de Classificação:")
print(classification_report(y_teste, pred_dt))
print("Matriz de Confusão:")
print(confusion_matrix(y_teste, pred_dt))

Acurácia Global no Teste: 92.31%

Relatório de Classificação:
              precision    recall  f1-score   support

  alto risco       0.88      1.00      0.93         7
 baixo risco       1.00      0.83      0.91         6

    accuracy                           0.92        13
   macro avg       0.94      0.92      0.92        13
weighted avg       0.93      0.92      0.92        13

Matriz de Confusão:
[[7 0]
 [1 5]]


> **Análise Específica do Modelo 2 (Árvore de Decisão):**
> A Árvore de Decisão (`max_depth=5`) também obteve **92,31% de acurácia global (12 acertos em 13)**, porém com perfil de erro oposto ao da Regressão Logística. O seu maior destaque clínico foi o **Recall de 100% para Alto Risco** (capturou todos os 7 pacientes graves, com zero falsos negativos). O seu único erro foi 1 Falso Positivo (encaminhou 1 caso de baixo risco como prioritário ao PS), o que é seguro em triagens emergenciais pois nenhuma vida corre risco desassistida. Contudo, sua limitação técnica em NLP decorre da divisão binária rígida por palavras-chave: a árvore não fornece probabilidades contínuas suaves e tende a sobreajustar (*overfitting*) quando exposta a variações vocabulares mais complexas.

### 7. Síntese Comparativa e Decisão de Engenharia

Embora ambos os modelos empatem na acurácia global (92,31%), a **Regressão Logística foi selecionada como o motor oficial do portal em produção**. Sua capacidade de calcular probabilidades contínuas viabiliza a exibição da porcentagem de confiança ao médico e possibilita a calibração do limiar de risco conforme o protocolo hospitalar adotado.

### 8. Teste Prático em Frases Inéditas (Predição em Tempo Real com Probabilidades)

In [7]:
frases_novas = [
    "Estou sentindo uma dor muito forte no meio do peito e falta de ar",
    "Tive apenas uma leve pontada muscular nas costas apos a academia",
    "Aperto insuportavel no torax com suor frio e sensacao de desmaio",
    "Desconforto nos ombros por ficar muito tempo sentado trabalhando"
]

frases_vec = vetorizador.transform(frases_novas)
predicoes = modelo_lr.predict(frases_vec)
probabilidades = modelo_lr.predict_proba(frases_vec)

print("=== RESULTADOS DA PREDIÇÃO EM TEMPO REAL ===")
for frase, pred, prob in zip(frases_novas, predicoes, probabilidades):
    conf = prob.max() * 100
    print(f"\nFrase: \"{frase}\"")
    print(f"--> Classificação: {pred.upper()} (Confiança: {conf:.1f}%)")

=== RESULTADOS DA PREDIÇÃO EM TEMPO REAL ===

Frase: "Estou sentindo uma dor muito forte no meio do peito e falta de ar"
--> Classificação: ALTO RISCO (Confiança: 67.5%)

Frase: "Tive apenas uma leve pontada muscular nas costas apos a academia"
--> Classificação: BAIXO RISCO (Confiança: 70.4%)

Frase: "Aperto insuportavel no torax com suor frio e sensacao de desmaio"
--> Classificação: ALTO RISCO (Confiança: 60.7%)

Frase: "Desconforto nos ombros por ficar muito tempo sentado trabalhando"
--> Classificação: BAIXO RISCO (Confiança: 58.5%)
